# Análisis de Capturas Pesqueras mediante Modelos de Machine Learning

In [2]:
#INSTALACIÓN DE DEPENDENCIAS
!pip install -q xgboost statsmodels scikit-learn pandas matplotlib seaborn plotly umap-learn

/bin/bash: línea 1: pip: orden no encontrada


In [3]:
# IMPORTS Y CONFIGURACIÓN GENERAL
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

# XGBoost
from xgboost import XGBRegressor

# Series de tiempo
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Configuración visual
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# Semilla para reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


# FASE 1 — COMPRENSIÓN DEL NEGOCIO

**Problema:** Los organismos de gestión pesquera en Argentina carecen de
herramientas predictivas para estimar volúmenes de captura por especie,
puerto y flota, lo que genera incertidumbre en la logística portuaria
y la asignación de recursos operativos.

**Objetivo medible:** Desarrollar modelos predictivos que estimen la
captura mensual (en kg) a nivel de registro (especie × puerto × flota × mes),
reduciendo al menos un 15% el MAE respecto del baseline naïve (captura del
mismo mes del año anterior para el mismo grupo).

**Estrategia de modelado:**
- **Modelo global:** un único modelo con todos los registros.
- **Modelos segmentados por especie agrupada:** un modelo independiente
  por cada especie del Top 5 por volumen.
- **Comparación:** ambos enfoques evaluados sobre el mismo test (2019).

*Justificación de segmentar por especie:* cada especie tiene ciclos
biológicos y estacionalidades propias. Se validará en el EDA que los
perfiles mensuales difieren significativamente entre especies.

**KPIs del modelo:**

| Métrica | Umbral de éxito |
|---------|----------------|
| MAE     | Reducción ≥ 15% vs. baseline |
| MAPE    | < 50% |
| R²      | > 0.5 |

**Hipótesis:**
1. Existen patrones estacionales diferenciados por especie → validar con
   perfil mensual y test de Kruskal-Wallis
2. La ubicación geográfica influye en el volumen → validar con importancia
   de features (lat/lon)
3. El tipo de flota impacta la captura → validar con importancia de
   features (flota)

**Restricciones:**
- Período: 2010-01 a 2019-11 (119 meses; falta diciembre 2019)
- Sin variables exógenas (clima, cuotas, precios)
- 5.85% de coordenadas geográficas nulas
- Series irregulares por grupo (no todos tienen registros mensuales continuos)
- Target muy asimétrico: media ~170K kg vs. mediana ~2.8K kg

**Validación temporal:**
- Train: 2010–2018 | Test: 2019
- El test set NO se usa para ajustar hiperparámetros ni early stopping

In [4]:
df = pd.read_csv('data/captura-puerto-flota-2010-2019.csv')

print(f"Dimensiones del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")

# Verificación básica de integridad
print(f"\nRegistros duplicados: {df.duplicated().sum():,}")
print(f"Rango de fechas: {df['fecha'].min()} a {df['fecha'].max()}")

# Cada fila = una combinación de especie × puerto × flota × mes
# La variable 'captura' está expresada en kilogramos
df.head(10)

Dimensiones del dataset: 44,380 filas × 13 columnas
Columnas: ['fecha', 'flota', 'puerto', 'provincia', 'provincia_id', 'departamento', 'departamento_id', 'latitud', 'longitud', 'categoria', 'especie', 'especie_agrupada', 'captura']

Registros duplicados: 0
Rango de fechas: 2010-01 a 2019-11


,fecha,flota,puerto,provincia,provincia_id,departamento,departamento_id,latitud,longitud,categoria,especie,especie_agrupada,captura
0,2010-01,Costeros,Caleta Cordova,Chubut,26,Escalante,26021,-45.748762,-67.377537,Peces,Merluza hubbsi,Merluza hubbsi S41,386114
1,2010-01,Costeros,Caleta Cordova,Chubut,26,Escalante,26021,-45.748762,-67.377537,Peces,Pez gallo,otras especies,4367
2,2010-01,Costeros,Caleta Cordova,Chubut,26,Escalante,26021,-45.748762,-67.377537,Peces,Rayas nep,Rayas (sin V. Cost),13
3,2010-01,Rada o ría,Caleta Cordova,Chubut,26,Escalante,26021,-45.748762,-67.377537,Crustáceos,Centolla,Centolla,48218
4,2010-01,Rada o ría,Caleta Cordova,Chubut,26,Escalante,26021,-45.748762,-67.377537,Peces,Merluza hubbsi,Merluza hubbsi S41,935
5,2010-01,Costeros,Caleta Olivia / Paula,Santa Cruz,78,Deseado,78014,-46.436049,-67.514904,Crustáceos,Centolla,Centolla,6
6,2010-01,Costeros,Caleta Olivia / Paula,Santa Cruz,78,Deseado,78014,-46.436049,-67.514904,Peces,Merluza hubbsi,Merluza hubbsi S41,73639
7,2010-01,Costeros,Caleta Olivia / Paula,Santa Cruz,78,Deseado,78014,-46.436049,-67.514904,Peces,Otras especies de peces,otras especies,6087
8,2010-01,Costeros,Caleta Olivia / Paula,Santa Cruz,78,Deseado,78014,-46.436049,-67.514904,Peces,Rayas nep,Rayas (sin V. Cost),16641
9,2010-01,Rada o ría,Caleta Olivia / Paula,Santa Cruz,78,Deseado,78014,-46.436049,-67.514904,Crustáceos,Centolla,Centolla,62087


In [6]:
# INSPECCIÓN INICIAL DEL DATASET
print("=" * 60)
print("INFORMACIÓN GENERAL")
print("=" * 60)
df.info()

# Estadísticas solo de variables numéricas relevantes
print("\n" + "=" * 60)
print("ESTADÍSTICAS DESCRIPTIVAS — VARIABLE OBJETIVO")
print("=" * 60)
stats_captura = df['captura'].describe()
print(stats_captura)
print(f"\nAsimetría (skewness): {df['captura'].skew():.2f}")
print(f"Ratio media/mediana:  {df['captura'].mean() / df['captura'].median():.1f}x")
print(f"Registros con captura = 0: {(df['captura'] == 0).sum()}")
print(f"Registros con captura < 0: {(df['captura'] < 0).sum()}")

# Nota: provincia_id y departamento_id son códigos, no variables numéricas.
# No se incluyen en el análisis descriptivo.

# Coordenadas geográficas
print("\n" + "=" * 60)
print("ESTADÍSTICAS — COORDENADAS GEOGRÁFICAS")
print("=" * 60)
display(df[['latitud', 'longitud']].describe())

# Valores nulos
print("\n" + "=" * 60)
print("VALORES NULOS POR COLUMNA")
print("=" * 60)
nulos = df.isnull().sum()
nulos_pct = (df.isnull().sum() / len(df) * 100).round(2)
tabla_nulos = pd.DataFrame({'Nulos': nulos, '% Nulos': nulos_pct})
display(tabla_nulos[tabla_nulos['Nulos'] > 0])

print(f"\n Las coordenadas nulas corresponden a puertos sin "
      f"georreferenciación ({nulos['latitud']:,} registros, {nulos_pct['latitud']}%)")

INFORMACIÓN GENERAL
<class 'pandas.DataFrame'>
RangeIndex: 44380 entries, 0 to 44379
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   fecha             44380 non-null  str    
 1   flota             44380 non-null  str    
 2   puerto            44380 non-null  str    
 3   provincia         44380 non-null  str    
 4   provincia_id      44380 non-null  int64  
 5   departamento      44380 non-null  str    
 6   departamento_id   44380 non-null  int64  
 7   latitud           41784 non-null  float64
 8   longitud          41784 non-null  float64
 9   categoria         44380 non-null  str    
 10  especie           44380 non-null  str    
 11  especie_agrupada  44380 non-null  str    
 12  captura           44380 non-null  int64  
dtypes: float64(2), int64(3), str(8)
memory usage: 4.4 MB

ESTADÍSTICAS DESCRIPTIVAS — VARIABLE OBJETIVO
count    4.438000e+04
mean     1.696445e+05
std      9.287020e+05
min   

,latitud,longitud
count,41784.000000,41784.000000
mean,-39.970128,-60.417967
std,3.763685,3.858639
min,-54.869566,-68.413519
25%,-40.725698,-64.934194
50%,-38.049150,-57.536848
75%,-38.049150,-57.536848
max,-35.745949,-56.746143



VALORES NULOS POR COLUMNA


,Nulos,% Nulos
latitud,2596,5.85
longitud,2596,5.85



 Las coordenadas nulas corresponden a puertos sin georreferenciación (2,596 registros, 5.85%)


In [7]:
# CARDINALIDAD DE VARIABLES CATEGÓRICAS
categoricas = ['flota', 'puerto', 'provincia', 'categoria', 'especie', 'especie_agrupada']

for col in categoricas:
    n_unique = df[col].nunique()
    print(f"\n{'='*50}")
    print(f"{col.upper()} — {n_unique} valores únicos")
    print(f"{'='*50}")
    print(df[col].value_counts().head(10))

# Resumen para decisiones de modelado
print("\n" + "=" * 60)
print("RESUMEN DE CARDINALIDAD PARA MODELADO")
print("=" * 60)
print(f"""
- especie: 90 valores únicos → Demasiados para segmentar
- especie_agrupada: 16 valores → Adecuada para segmentación
  (se usará esta variable para los modelos por especie)
- flota: 8 valores únicos
- puerto: 23 valores únicos
- provincia: 6 valores (incluye 'sin especificar': {(df['provincia']=='sin especificar').sum()} registros)
- categoria: 4 valores (incluye 'otras': {(df['categoria']=='otras').sum()} registro)
""")

# Top 5 especies agrupadas por volumen (las que se usarán para segmentar)
top5_vol = (df.groupby('especie_agrupada')['captura']
            .sum()
            .sort_values(ascending=False)
            .head(5))

print("TOP 5 ESPECIES AGRUPADAS POR VOLUMEN TOTAL (candidatas para segmentación):")
for i, (esp, vol) in enumerate(top5_vol.items(), 1):
    n_registros = (df['especie_agrupada'] == esp).sum()
    print(f"  {i}. {esp}: {vol/1e6:.1f} M kg ({n_registros:,} registros)")


FLOTA — 8 valores únicos
flota
Rada o ría                         13349
Costeros                           13016
Fresqueros                         10184
Congeladores arrastreros            6198
Congeladores tangoneros              935
Congeladores poteros nacionales      307
Congeladores trampas                 196
Congeladores palangreros             195
Name: count, dtype: int64

PUERTO — 23 valores únicos
puerto
Mar del Plata                 19117
Necochea / Quequén             4658
San Antonio Oeste              3457
Puerto Madryn                  2676
otros puertos Buenos Aires     2508
General Lavalle                2265
San Antonio Este               1829
Ushuaia                        1275
Puerto Deseado                 1105
Rawson                         1089
Name: count, dtype: int64

PROVINCIA — 6 valores únicos
provincia
Buenos Aires        30594
Río Negro            5306
Chubut               5025
Santa Cruz           2088
Tierra del Fuego     1279
sin especificar        